# Chapter 22 — Time Series I: Forecasting Inflation

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch22_forecasting_inflation.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 22.1 A monthly series with a date index

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

from indiadata import mospi_cpi

cpi = mospi_cpi(sector="combined").set_index("date").asfreq("MS")
print(cpi.index.freqstr, "|", cpi.index.min().date(), "to", cpi.index.max().date(),
      "|", len(cpi), "months")
cpi[["index", "inflation", "status"]].tail(3)

In [ ]:
cpi.loc["2020-03":"2020-06", ["index", "inflation", "status"]]

In [ ]:
pi = cpi["inflation"]["2014-01":].copy()             # y-o-y %, as published
lp = 100 * np.log(cpi["index"])                      # 100 × log price level
lp.loc["2020-04":"2020-05"] = np.nan                 # imputed: treat as missing
print("Missing inflation months:", list(pi[pi.isna()].index.strftime("%b %Y")))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.plot(pi.index, pi, color="#1f4e79", lw=1.6)
ax.axvspan(pd.Timestamp("2022-01-01"), pi.index[-1], color="#9db7d5", alpha=0.25)
ax.text(pd.Timestamp("2022-03-01"), pi.max() * 0.95, "test period", fontsize=9,
        color="#1f4e79")
ax.axvspan(pd.Timestamp("2020-03-15"), pd.Timestamp("2020-05-15"), color="#c55a11",
           alpha=0.4)
ax.annotate("no data\nApr–May 2020", xy=(pd.Timestamp("2020-04-15"), 2.2),
            xytext=(pd.Timestamp("2018-01-01"), 1.0), fontsize=9,
            arrowprops=dict(arrowstyle="->", color="grey"))
ax.axhline(4, color="grey", ls="--", lw=1)
ax.set_ylabel("Year-on-year, %")
ax.set_title("CPI-Combined inflation, India, 2014–2025", loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.03, "Dashed line: RBI target of 4%. Source: MoSPI, CPI (base 2012=100), "
         "eSankhyiki API.", fontsize=8, color="grey")
plt.show()

## 22.2 Lags, differences and autocorrelation

In [ ]:
check = pd.DataFrame({"inflation": pi,
                      "lag 1": pi.shift(1),
                      "change": pi.diff(),
                      "from log index": (lp.diff(12))["2014-01":]})
check.loc["2025-09":"2025-12"].round(2)

In [ ]:
pre = pi["2014-01":"2019-12"]
mom = lp.diff()["2014-01":"2019-12"]            # month-on-month % (log points)

fig, axes = plt.subplots(2, 2, figsize=(10, 6.5))
plot_acf(pre, lags=24, ax=axes[0, 0], title="ACF: y-o-y inflation")
plot_pacf(pre, lags=24, ax=axes[0, 1], title="PACF: y-o-y inflation", method="ywm")
plot_acf(mom, lags=24, ax=axes[1, 0], title="ACF: month-on-month change")
plot_pacf(mom, lags=24, ax=axes[1, 1], title="PACF: month-on-month change",
          method="ywm")
for ax in axes.flat:
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xticks([0, 6, 12, 18, 24])
    ax.title.set_fontsize(11)
fig.suptitle("Autocorrelations of Indian CPI inflation, 2014–2019", x=0.01,
             ha="left", fontsize=13)
fig.text(0.01, -0.01, "Shaded area: approximate 95% band under no autocorrelation. "
         "Source: MoSPI CPI-Combined.", fontsize=8, color="grey")
plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.tsa.stattools import acf
print("ACF of y-o-y inflation at lags 1, 6, 12:",
      acf(pre, nlags=12)[[1, 6, 12]].round(2))
print("ACF of month-on-month change at lags 1, 6, 12:",
      acf(mom, nlags=12)[[1, 6, 12]].round(2))

## 22.3 Stationarity: ADF and KPSS

In [ ]:
def unit_root_table(series_dict):
    rows = {}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")      # KPSS warns when p is off its table
        for name, x in series_dict.items():
            x = x.dropna()
            adf_stat, adf_p, *_ = adfuller(x, autolag="AIC")
            kpss_stat, kpss_p, *_ = kpss(x, regression="c", nlags="auto")
            rows[name] = {"obs": len(x), "ADF stat": adf_stat, "ADF p": adf_p,
                          "KPSS stat": kpss_stat, "KPSS p": kpss_p}
    return pd.DataFrame(rows).T.round(3)

unit_root_table({"log CPI level": lp["2014-01":],
                 "m-o-m change": lp.diff()["2014-01":],
                 "y-o-y inflation": pi,
                 "change in y-o-y inflation": pi.diff()})

## 22.4 Split by time: training and test samples

In [ ]:
TRAIN_END, TEST_START = "2021-12-01", "2022-01-01"
train = pi[:TRAIN_END]
test = pi[TEST_START:]
print(f"training: {train.index[0]:%b %Y}–{train.index[-1]:%b %Y}, {train.notna().sum()} months "
      f"with data;  test: {test.index[0]:%b %Y}–{test.index[-1]:%b %Y}, {len(test)} months")
print(f"Test-period inflation: mean {test.mean():.2f}%, min {test.min():.2f}% "
      f"({test.idxmin():%b %Y}), max {test.max():.2f}% ({test.idxmax():%b %Y})")

## 22.5 AR and ARIMA models, chosen by information criteria

In [ ]:
warnings.filterwarnings("ignore", module="statsmodels")   # see the note below

grid = []
for d in (0, 1):
    for p in range(0, 4):
        for q in range(0, 3):
            if p == q == 0:
                continue
            res = SARIMAX(train, order=(p, d, q), trend="c" if d == 0 else "n").fit(disp=False)
            grid.append({"p": p, "d": d, "q": q, "AIC": res.aic, "BIC": res.bic})
grid = pd.DataFrame(grid)
for d in (0, 1):
    print(f"Best models with d = {d}:")
    print(grid[grid["d"] == d].nsmallest(3, "AIC").round(1).to_string(index=False))

In [ ]:
best_ar = grid[(grid["d"] == 0) & (grid["q"] == 0)].nsmallest(1, "AIC").iloc[0]
best_i = grid[grid["d"] == 1].nsmallest(1, "AIC").iloc[0]
AR_ORDER = (int(best_ar["p"]), 0, 0)
ARIMA_ORDER = (int(best_i["p"]), 1, int(best_i["q"]))
print("AR model:", AR_ORDER, "  ARIMA model:", ARIMA_ORDER)

ar_fit = SARIMAX(train, order=AR_ORDER, trend="c").fit(disp=False)
print(ar_fit.summary().tables[1])

In [ ]:
phi = ar_fit.params.filter(like="ar.L")
print(f"Sum of AR coefficients: {phi.sum():.3f}")
print(f"Implied long-run mean of inflation: {ar_fit.params['intercept'] / (1 - phi.sum()):.2f}%")

In [ ]:
lb = acorr_ljungbox(ar_fit.resid.dropna()[AR_ORDER[0]:], lags=[6, 12, 24])
lb.round(3)

## 22.6 Seasonality: SARIMA

In [ ]:
seasonal = []
for order in (AR_ORDER, ARIMA_ORDER):
    for sorder in [(0, 0, 0, 12), (0, 0, 1, 12), (1, 0, 1, 12)]:
        res = SARIMAX(train, order=order, seasonal_order=sorder,
                      trend="c" if order[1] == 0 else "n").fit(disp=False)
        seasonal.append({"order": order, "seasonal": sorder[:3], "AIC": res.aic})
seasonal = pd.DataFrame(seasonal)
seasonal.round(1)

In [ ]:
best_s = seasonal.loc[seasonal["AIC"].idxmin()]
SAR_ORDER, SAR_SEASONAL = best_s["order"], tuple(best_s["seasonal"]) + (12,)
sar_fit = SARIMAX(train, order=SAR_ORDER, seasonal_order=SAR_SEASONAL,
                  trend="c" if SAR_ORDER[1] == 0 else "n").fit(disp=False)
print("SARIMA chosen:", SAR_ORDER, SAR_SEASONAL)
print(sar_fit.params.round(3).to_string())
print(acorr_ljungbox(sar_fit.resid.dropna()[13:], lags=[12, 24]).round(3))

In [ ]:
AIRLINE = dict(order=(0, 1, 1), seasonal_order=(0, 1, 1, 12))
air_fit = SARIMAX(lp[:TRAIN_END], **AIRLINE).fit(disp=False)
print(air_fit.params.round(3).to_string())


def lp_to_inflation(lp_forecast, lp_history):
    """Turn forecasts of 100*log CPI into y-o-y inflation (%) for the same months."""
    base = lp_history.reindex(lp_forecast.index - pd.DateOffset(months=12)).to_numpy()
    return pd.Series(100 * (np.exp((lp_forecast.to_numpy() - base) / 100) - 1),
                     index=lp_forecast.index)

## 22.7 Benchmarks: the random walk and the seasonal naive forecast

In [ ]:
o = pd.Timestamp(TRAIN_END)
months = pd.date_range(o + pd.DateOffset(months=1), periods=3, freq="MS")
snaive_lp = pd.Series([lp[o] + lp[m - pd.DateOffset(months=12)] - lp[o - pd.DateOffset(months=12)]
                       for m in months], index=months)
print("Seasonal naive, 100 × log y-o-y:", (snaive_lp - lp.reindex(months - pd.DateOffset(months=12)).to_numpy()).round(3).tolist())
print("Random walk,    100 × log y-o-y:", round(lp[o] - lp[o - pd.DateOffset(months=12)], 3))

## 22.8 Rolling-origin forecasts

In [ ]:
H = 3
origins = pd.date_range(TRAIN_END, "2025-11-01", freq="MS")
MODELS = {
    f"AR{AR_ORDER}": dict(data="pi", order=AR_ORDER, seasonal_order=(0, 0, 0, 0)),
    f"ARIMA{ARIMA_ORDER}": dict(data="pi", order=ARIMA_ORDER, seasonal_order=(0, 0, 0, 0)),
    "SARIMA": dict(data="pi", order=SAR_ORDER, seasonal_order=SAR_SEASONAL),
    "Airline (log CPI)": dict(data="lp", **AIRLINE),
}

rows = []
for origin in origins:
    rows.append({"origin": origin, "model": "Random walk",
                 **{h: pi[origin] for h in range(1, H + 1)}})
    for name, spec in MODELS.items():
        y_hist = (pi if spec["data"] == "pi" else lp)[:origin]
        trend = "c" if spec["data"] == "pi" and spec["order"][1] == 0 else "n"
        fit = SARIMAX(y_hist, order=spec["order"], seasonal_order=spec["seasonal_order"],
                      trend=trend).fit(disp=False)
        f = fit.forecast(H)
        if spec["data"] == "lp":
            f = lp_to_inflation(f, lp)
        rows.append({"origin": origin, "model": name,
                     **{h: f.iloc[h - 1] for h in range(1, H + 1)}})

fc = (pd.DataFrame(rows)
        .melt(id_vars=["origin", "model"], var_name="h", value_name="forecast"))
fc["h"] = fc["h"].astype(int)
fc["target"] = (fc["origin"].dt.to_period("M") + fc["h"]).dt.to_timestamp()
fc["actual"] = fc["target"].map(pi)
fc = fc.dropna(subset=["actual"])
fc["error"] = fc["actual"] - fc["forecast"]
print(fc.shape, "forecasts;", fc["target"].min().date(), "to", fc["target"].max().date())

## 22.9 Did ARIMA beat the benchmark?

In [ ]:
acc = fc.groupby(["h", "model"])["error"].agg(
    RMSE=lambda e: np.sqrt((e ** 2).mean()), MAE=lambda e: e.abs().mean(), n="size")
acc["RMSE vs RW"] = acc["RMSE"] / acc["RMSE"].xs("Random walk", level="model").reindex(
    acc.index.get_level_values("h")).to_numpy()
acc.round(3)

In [ ]:
q_end = fc[fc["origin"].dt.month.isin([3, 6, 9, 12])]
q_avg = (q_end.groupby(["origin", "model"])[["forecast", "actual"]].mean()
              .query("@q_end.groupby(['origin', 'model']).size() == 3"))
q_err = (q_avg["actual"] - q_avg["forecast"]).groupby(level="model")
quarterly = pd.DataFrame({"RMSE": q_err.apply(lambda e: np.sqrt((e ** 2).mean())),
                          "MAE": q_err.apply(lambda e: e.abs().mean()),
                          "quarters": q_err.size()})
quarterly["RMSE vs RW"] = quarterly["RMSE"] / quarterly.loc["Random walk", "RMSE"]
quarterly.sort_values("RMSE").round(3)

## 22.10 Is the difference significant? A Diebold–Mariano test

In [ ]:
def diebold_mariano(e1, e2, h):
    """DM test of equal MSE (HLN small-sample version). Positive DM: model 2 better."""
    d = np.asarray(e1) ** 2 - np.asarray(e2) ** 2
    n = len(d)
    dc = d - d.mean()
    lrv = dc @ dc / n + 2 * sum(dc[k:] @ dc[:-k] / n for k in range(1, h))
    dm = d.mean() / np.sqrt(lrv / n)
    dm *= np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)      # HLN correction
    return dm, 2 * stats.t.sf(abs(dm), df=n - 1)


out = []
for h in (1, 3):
    sub = fc[fc["h"] == h].pivot(index="target", columns="model", values="error")
    for m in ["SARIMA", "Airline (log CPI)", f"ARIMA{ARIMA_ORDER}"]:
        dm, p = diebold_mariano(sub["Random walk"], sub[m], h)
        out.append({"h": h, "model vs random walk": m, "DM": dm, "p-value": p})
pd.DataFrame(out).round(3)

## 22.11 The forecasts, month by month

In [ ]:
h3 = fc[fc["h"] == 3].pivot(index="target", columns="model", values="forecast")
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(test.index, test, color="black", lw=2, label="Actual inflation")
ax.plot(h3.index, h3["Random walk"], color="#9db7d5", lw=1.6, label="Random walk")
ax.plot(h3.index, h3["SARIMA"], color="#c55a11", lw=1.6, label="SARIMA")
ax.plot(h3.index, h3[f"AR{AR_ORDER}"], color="#1f4e79", lw=1.2, ls="--",
        label=f"AR{AR_ORDER}")
ax.axhline(4, color="grey", lw=0.8, ls=":")
ax.set_ylabel("Year-on-year, %")
ax.set_title("Forecasting three months ahead, 2022–2025", loc="left", fontsize=13)
ax.legend(frameon=False, ncol=4, loc="upper right", fontsize=9)
ax.set_ylim(top=ax.get_ylim()[1] + 1)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.03, "Each forecast is plotted at the month it targets, made 3 months "
         "earlier with data available then. Source: MoSPI CPI; author's models.",
         fontsize=8, color="grey")
plt.show()

In [ ]:
worst = (fc[(fc["h"] == 3) & (fc["model"] == "SARIMA")]
         .assign(abs_err=lambda x: x["error"].abs()).nlargest(3, "abs_err"))
worst[["origin", "target", "forecast", "actual", "error"]].round(
    {"forecast": 2, "actual": 2, "error": 2})

In [ ]:
from indiadata import mospi_cpi_groups

veg = (mospi_cpi_groups().query("subgroup == 'Vegetables'")
                         .set_index("date")["inflation"])
pd.DataFrame({"vegetables at origin": veg.reindex(worst["origin"]).to_numpy(),
              "vegetables at target": veg.reindex(worst["target"]).to_numpy()},
             index=worst["target"].dt.strftime("%b %Y")).round(1)

## 22.12 Forecast intervals and a fan chart

In [ ]:
final = SARIMAX(pi, order=SAR_ORDER, seasonal_order=SAR_SEASONAL,
                trend="c" if SAR_ORDER[1] == 0 else "n").fit(disp=False)
fcast = final.get_forecast(12)
central = fcast.predicted_mean
bands = {lev: fcast.conf_int(alpha=1 - lev / 100) for lev in (50, 80, 95)}
summary = pd.DataFrame({"central": central,
                        "95% low": bands[95].iloc[:, 0], "95% high": bands[95].iloc[:, 1]})
summary.iloc[[0, 2, 5, 11]].round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
hist = pi["2023-01":]
ax.plot(hist.index, hist, color="black", lw=1.8, label="Actual")
for lev, alpha in [(95, 0.18), (80, 0.3), (50, 0.45)]:
    lo, hi = bands[lev].iloc[:, 0], bands[lev].iloc[:, 1]
    ax.fill_between(central.index, lo, hi, color="#c55a11", alpha=alpha, lw=0,
                    label=f"{lev}% interval")
ax.plot(central.index, central, color="#c55a11", lw=2, label="Central forecast")
ax.axhspan(2, 6, color="tab:green", alpha=0.06)
ax.axhline(4, color="grey", ls="--", lw=0.8)
ax.set_ylabel("Year-on-year, %")
ax.set_title("SARIMA fan chart for CPI inflation, 2026", loc="left", fontsize=13)
ax.legend(frameon=False, loc="upper center", ncol=3, fontsize=9)
ax.set_ylim(top=ax.get_ylim()[1] + 1.5)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.03, "Model estimated on Jan 2014–Dec 2025. Green band: RBI tolerance "
         "band 2–6%. Intervals assume normal shocks and a correct model.",
         fontsize=8, color="grey")
plt.show()

In [ ]:
w95 = bands[95].iloc[:, 1] - bands[95].iloc[:, 0]
print(f"Width of the 95% interval: {w95.iloc[0]:.1f} points at 1 month, "
      f"{w95.iloc[2]:.1f} at 3 months, {w95.iloc[11]:.1f} at 12 months")
cover = (fc[fc["model"] == "SARIMA"].groupby("h")["error"]
           .apply(lambda e: (e.abs() <= 1.96 * np.sqrt(final.params["sigma2"])).mean()))
print("Share of test errors within ±1.96 × the one-step SD:", cover.round(2).to_dict())

## 22.13 Verdict

## Key ideas

## Exercises

## Data citation

## Further reading